## Settings

In [ ]:
# Install and load all packages
#=========================================================

if (!require("pacman")) install.packages("pacman")
pacman::p_load("terra", "sf", "corrplot", "cluster","here",
  "tibble", "dplyr", "httr", "jsonlite",
  "stringr", "purrr", "lubridate", "tidyr"
)

# Include precipitation in this run?
param_include_precipitation <- 1  # 1 = include, 0 = soil only



In [ ]:
secret_knmi_api_key <- "API key"

## Veluwe Polygon

In [ ]:
## Read in the Natura2000 shapefile
natura2000_sf <- sf::read_sf("https://service.pdok.nl/rvo/natura2000/atom/downloads/natura2000.gpkg") |>
  sf::st_transform(28992)

# Create the Veluwe sf by filtering the Natura2000 shapefile 
veluwe_sf <- natura2000_sf|>
  dplyr::filter(objectid == 4049)

veluwe_v <- vect(veluwe_sf)

# Load land use map
lgn5 <- terra::rast("/home/jovyan/Cloud Storage/naa-vre-public/vl-veluwe-forest-model/LGN5.tif")

# Crop national map to Veluwe using the Natura2000 polygon
lgn5_veluwe <- terra::crop(lgn5, veluwe_v)
lgn5_veluwe <- terra::mask(lgn5_veluwe, veluwe_v)


# Retain only the forest (11 and 12)
forest_codes <- c(11, 12)
veluwe_forest <- ifel(lgn5_veluwe %in% forest_codes, 1, NA)

# Turn raster into a polygon 
veluwe_forest_v <- as.polygons(veluwe_forest,aggregate = TRUE,na.rm = TRUE)
terra::plot(veluwe_forest_v)

# Outputs:
veluwe_file <- "/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/veluwe_forest_v.gpkg"

writeVector(
  veluwe_forest_v,
  veluwe_file,
  overwrite = TRUE
)

## Clustering input: environmental data

In [ ]:
##DOWNLOAD ENVIRONMENTAL DATA
data_dir <- "/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/env_layers"
dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)

urls <- c(
  clay  = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/1cda77df-c907-49a9-870f-b48ac84bd63f",
  sand  = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/63bfd9d8-e2a6-46d8-876e-dae74f34a89d",
  silt  = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/4ce01d2d-ddda-4750-ac03-e5208e1dcd4a",
  CEC   = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/2f7365fe-5587-4c65-91a9-259131197b1b",
  pH    = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/447dd6ba-4a5f-4006-9c68-f3c3b4452206",
  SOM   = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/3bd552f0-2efa-49dd-ad5b-a184a516d906",
  N     = "https://data.4tu.nl/file/0c934ac6-2e95-4422-8360-d3a802766c71/56e66069-806b-46d0-bd70-c07e6e340744"
)

# Increase time out time for such large download
options(timeout = 12 * 60 * 60)

# Get the file path for all urls and then dowload and unzip
for (name in names(urls)) {
  zip_file <- file.path(data_dir, paste0(name, ".zip"))

  # Download only when the file does not already exist
  if (!file.exists(zip_file)) {
    download.file(urls[[name]],zip_file, mode = "wb", )
  } 
    
suppressWarnings(
    unzip(zip_file, exdir = data_dir, overwrite = FALSE)
)
}

##READ ALL FILES IN ONE GO

# Get a list of files that include clay, silt and sand
files_texture <- list.files(data_dir, pattern = "0_5.*_mean_processed\\.tif$", full.names = TRUE, recursive = TRUE ) 


# List of files that are not soil texture
files_rest <- list.files(data_dir, pattern = "(pH|N|CEC|SOM).*0_5.*_pred_mean\\.tif$", full.names = TRUE, recursive = TRUE ) 


# Combine both file lists
files_all <- c(files_texture, files_rest)

# Keep only SOM in 2000
files_all <- files_all[
  !grepl("SOM_per_", basename(files_all)) |
    grepl("SOM_per_2000", basename(files_all))
]

# Create a spatial raster of all the files
layers <- rast(files_all)

# Outputs:
env_layers_file <- "/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/env_layers_0_5.tif"

writeRaster(
  layers,
  env_layers_file,
  overwrite = TRUE
)

In [ ]:
##CROP DATA LAYERS TO INPUT MAP 
layers <- rast(env_layers_file)
veluwe_forest_v <- vect(veluwe_file)

layers_veluwe <- crop(layers, veluwe_forest_v)
layers_veluwe <- mask(layers_veluwe, veluwe_forest_v)

# Clean the names
names(layers_veluwe) <- c("clay", "sand", "silt","CEC", "pH", "SOM_2000", "N_total")

# Outputs:
layers_veluwe_file <- "/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/layers_veluwe.tif"
writeRaster(layers_veluwe, layers_veluwe_file, overwrite = TRUE)



In [ ]:
Sys.setenv(KNMI_EDR_KEY = secret_knmi_api_key)

##LOAD PRECIPITATION DATA (OPTIONAL - Only run when using it in later cells)

# Load necessary input
layers_veluwe <- terra::rast(layers_veluwe_file)

#Use a prior made ecoregion map to get the veluwe bounding box
template <- terra::rast(layers_veluwe_file)

#Turn to vector as terra needs it for project(), also attatch correct CRS to polygon
cluster_extent <- terra::as.polygons(
  terra::ext(template),
  crs = terra::crs(template)
)

#Transform to coordinate system
cluster_extent_ll <- terra::project(
  cluster_extent,
  "EPSG:4326"
)

#Extract the extent
e <- terra::ext(cluster_extent_ll)

#Create a bounding box to 'crop' precipitation data
bbox <- c(
  e$xmin,
  e$ymin,
  e$xmax,
  e$ymax
)


#Load precipitation data
#Key
#Function to extract
retrieve_knmi_edr_data <- function(bbox,
                                   variable = c("mean temperature", "max temperature",
                                                "min temperature", "precipitation"),
                                   start_date,
                                   start_time = "00:00:00",
                                   end_date,
                                   end_time = "23:59:59",
                                   knmi_edr_key = Sys.getenv("KNMI_EDR_KEY")) {
  
  # Create KNMI variable lookup table to match variable inputs to KNMI collection & parameter names
  knmi_var_lookup <- tibble::tibble(
    collection = c("Tg1", "Tx1", "Tn1", "Rd1"),
    parameter = c("temperature", "temperature", "temperature", "precipitation"),
    var_name = c("mean temperature", "max temperature", "min temperature", "precipitation")
  )
  
  if(!(variable %in% knmi_var_lookup$var_name)) {
    
    stop("The weather variable you provided does not exist. Select one of: 'mean temperature', 'max temperature', 'min temperature', or 'precipitation'.")
    
  }
  
  # Check that start_date and end_date are of class 'Date' to ensure that we can retrieve
  if(!any(class(start_date) == "Date", class(end_date) == "Date")) {
    
    stop("Please provide dates as 'yyyy-mm-dd'.")
    
  }
  
  # If retrieving data from KNMI EDR API fails, try again
  
  # Send GET request to KNMI EDR API
  edr_get <- httr::RETRY(
    verb = "GET", url = paste0("https://api.dataplatform.knmi.nl/edr/v1/collections/",
                               knmi_var_lookup |> dplyr::filter(var_name == variable) |> dplyr::pull("collection"),
                               "/cube?bbox=", paste(bbox, collapse = "%2C"),
                               "&z=0",
                               "&parameter-name=",
                               knmi_var_lookup |> dplyr::filter(var_name == variable) |> dplyr::pull("parameter"),
                               "&datetime=",
                               start_date, "T",
                               stringr::str_replace_all(string = start_time,
                                                        pattern = ":",
                                                        replacement = "%3A"), "Z%2F",
                               end_date, "T",
                               stringr::str_replace_all(string = end_time,
                                                        pattern = ":",
                                                        replacement = "%3A"), "Z"),
    httr::add_headers(Authorization = knmi_edr_key),
    times = 6,
    pause_base = 2,
    pause_cap = 30 )
  
  httr::stop_for_status(edr_get)
  
  # Convert from JSON to R readable format
  edr_data <- jsonlite::fromJSON(txt = rawToChar(x = edr_get$content))
  
  
  
  
  
  # Output
  return(edr_data)
  
}


#Load the data for 2001-2005

precip <- purrr::map(.x = 2001:2005,
                     .f = ~{
                       
                       # Retrieve data for period 1 (1 Jan to 1 March)
                       period1 <- retrieve_knmi_edr_data(bbox = bbox,
                                                         variable = "precipitation",
                                                         start_date = lubridate::make_date(.x, 1, 1),
                                                         start_time = "00:00:00",
                                                         end_date = lubridate::make_date(.x, 3, 1),
                                                         end_time = "23:59:59")
                       
                       # Retrieve data for period 2 (2 March to 31 May)
                       period2 <- retrieve_knmi_edr_data(bbox = bbox,
                                                         variable = "precipitation",
                                                         start_date = lubridate::make_date(.x, 3, 2),
                                                         start_time = "00:00:00",
                                                         end_date = lubridate::make_date(.x, 5, 31),
                                                         end_time = "23:59:59")
                       
                       # Retrieve data for period 3 (1 Jun to 31 August)
                       period3 <- retrieve_knmi_edr_data(bbox = bbox,
                                                         variable = "precipitation",
                                                         start_date = lubridate::make_date(.x,  6, 1),
                                                         start_time = "00:00:00",
                                                         end_date = lubridate::make_date(.x, 8, 31),
                                                         end_time = "23:59:59")
                       
                       # Retrieve data for period 4 ( 1 September to 30 November )
                       
                       period4 <- retrieve_knmi_edr_data(bbox = bbox,
                                                         variable = "precipitation",
                                                         start_date = lubridate::make_date(.x, 9,1 ),
                                                         start_time = "00:00:00",
                                                         end_date = lubridate::make_date(.x, 11, 30),
                                                         end_time = "23:59:59")
                       # Retrieve data for period 5 ( 1 December to 31 December )
                       period5 <- retrieve_knmi_edr_data(
                         bbox = bbox,
                         variable = "precipitation",
                         start_date = lubridate::make_date(.x, 12, 1),
                         start_time = "00:00:00",
                         end_date = lubridate::make_date(.x, 12, 31),
                         end_time = "23:59:59"
                       )
                       
                       
                       # Match precipitation data to dates and coordinates
                       data1 <- tidyr::expand_grid(date = period1$domain$axes$t$values,
                                                   y = period1$domain$axes$y$values,
                                                   x = period1$domain$axes$x$values) |>
                         dplyr::mutate(precipitation = period1$ranges$precipitation$values)
                       
                       data2 <- tidyr::expand_grid(date = period2$domain$axes$t$values,
                                                   y = period2$domain$axes$y$values,
                                                   x = period2$domain$axes$x$values) |>
                         dplyr::mutate(precipitation = period2$ranges$precipitation$values)
                       
                       data3<- tidyr::expand_grid(date = period3$domain$axes$t$values,
                                                  y = period3$domain$axes$y$values,
                                                  x = period3$domain$axes$x$values) |>
                         dplyr::mutate(precipitation = period3$ranges$precipitation$values)
                       
                       
                       data4 <- tidyr::expand_grid(date = period4$domain$axes$t$values,
                                                   y = period4$domain$axes$y$values,
                                                   x = period4$domain$axes$x$values) |>
                         dplyr::mutate(precipitation = period4$ranges$precipitation$values)
                      
                       data5 <- tidyr::expand_grid(date = period5$domain$axes$t$values,
                                                   y = period5$domain$axes$y$values,
                                                   x = period5$domain$axes$x$values) |>
                         dplyr::mutate(precipitation = period5$ranges$precipitation$values)
                       
                       # Bind output of all periods
                       data <- dplyr::bind_rows(data1, data2, data3, data4,data5)
                       
                       return(data)
                       
                     },
                     .progress = TRUE) |>
  purrr::list_c()

#Save and read precipitation data
precipitation_file <- "/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/precipitation_Jan2001_to_December2005.csv"
write.csv(precip, precipitation_file, row.names = FALSE)



In [ ]:
## ADD PRECIPITATION (OPTIONAL)
# Add a run suffix to keep outcomes of clustering with and without precipitation separate
run_suffix <- if (param_include_precipitation == 1) "_with_precipitation" else "_soil_only"

# Load the veluwe polygon
veluwe_forest_v <- terra::vect(veluwe_file)
    
if (param_include_precipitation== 1) {
  # Read the precipitation table dowloaded in previous cell
  precip_data <- read.csv(precipitation_file)

#Calculate average across years per location
 precip_mean <- precip_data |>
  dplyr::group_by(x, y) |>
  dplyr::summarise(
    mean_daily_precipitation = mean(precipitation, na.rm = TRUE),
    .groups = "drop"
  )

  # Convert to a raster, as in the original precipitation script.
  precip_mean_raster <- terra::rast(
    precip_mean[, c("x", "y", "mean_daily_precipitation")],
    type = "xyz",
    crs = "EPSG:4326"
  )
  names(precip_mean_raster) <- "mean_daily_precipitation"

  # Project and align precipitation with the soil layers.
  precip_veluwe <- terra::project(
    precip_mean_raster,
    layers_veluwe[[1]],
    method = "bilinear"
  )
  precip_veluwe <- terra::mask(precip_veluwe, veluwe_forest_v)

  predictor_layers <- c(layers_veluwe, precip_veluwe)
} else {
  predictor_layers <- layers_veluwe
}

# Use the same dataframe name as the rest of the notebook.
layers_df <- as.data.frame(predictor_layers, cells = TRUE, na.rm = TRUE)
layers_df_file <- paste0(
  "/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/layers_veluwe",
  run_suffix, ".csv"
)
write.csv(layers_df, layers_df_file, row.names = FALSE)


In [ ]:
##CORRELATION MATRIX 
#Read in layers file
layers_df <- read.csv(layers_df_file)
cor_vars <- c("clay", "sand", "silt", "SOM_2000", "CEC", "N_total", "pH")
if (param_include_precipitation == 1) cor_vars <- c(cor_vars, "mean_daily_precipitation")

correlation_matrix <-  cor(
  layers_df[, cor_vars],
  use = "pairwise.complete.obs",
  method = "pearson"
)

# Plot the correlation matrix
corrplot(correlation_matrix, method = "color",type = "upper", order = "hclust", 
         tl.col = "black", tl.srt = 45,
         addCoef.col = "black", 
         number.cex = 0.7,)
#sand and silt -0-99 hence remove silt
# Keep only other columns


## PCA

In [ ]:
# Add a run suffix to keep outcomes of clustering with and without precipitation separate
run_suffix <- if (param_include_precipitation == 1L) "_with_precipitation" else "_soil_only"
#Read in layers df
layers_df <- read.csv(layers_df_file)

#Select variables applicable for PCA
pca_vars <- c("clay", "sand", "SOM_2000", "CEC", "N_total", "pH")
if (param_include_precipitation == 1 ) pca_vars <- c(pca_vars, "mean_daily_precipitation")
# Check for and remove NAs
colSums(is.na(layers_df))
layers_df <- layers_df[complete.cases(layers_df[, pca_vars]), ]

# Normalize the data for use in PCA. Subtracts the mean and divides by the sd from each value.
data_normalized <- scale(layers_df[, pca_vars])

# Carry out PCA
data.pca <- prcomp(data_normalized)

plot(data.pca$sdev^2, type="b") 

# Visualise the pca output
biplot(data.pca, scale= 0, var.axes= TRUE)
# See how much of the variance is explained by each principal component
summary(data.pca)
# Look at PC1,2 and 3 and round to 2dp. Can be used to check which variables should be kept
round(data.pca$rotation[, 1:3], 2)#all variables retained as all hadeigenvector loadings > 0.1 in the ﬁrst three principal components (Metzger et al.,2013)

pca_1_3 <- data.pca$x[, 1:3]

# Outputs:
#PCA scores
pca_scores_file <- paste0("/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/pca_scores", run_suffix, ".csv")
pca_scores <- data.frame(cell = layers_df$cell,pca_1_3)
write.csv(pca_scores, pca_scores_file, row.names = FALSE)

#Normalized data
data_normalized_file <- paste0("/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/normalized_data", run_suffix, ".csv")
write.csv(data_normalized, data_normalized_file, row.names = FALSE)

## Decide number of clusters

In [ ]:
##SILLHOUETTE 
#Decides k for PAM

# Add a run suffix to keep outcomes of clustering with and without precipitation separate
run_suffix <- if (param_include_precipitation == 1L) "_with_precipitation" else "_soil_only"

#Read in layers df
pca_scores <- read.csv(pca_scores_file)



# Exclude the cell column from clustering
pca_1_3 <- as.matrix(
  pca_scores[, c("PC1", "PC2", "PC3")]
)

# Make a sample 
set.seed(123)
pca_1_3_sample <- pca_1_3[
  sample(seq_len(nrow(pca_1_3)), size = min(5000, nrow(pca_1_3))),
]

# Compute average silhouette of observations for different values of k

k_values <- 2:8
silhouette_scores <- sapply(k_values, function(k) {
pam(pca_1_3_sample, k) $silinfo$avg.width})


plot(k_values, silhouette_scores, type = "b")
best_k <- k_values[which.max(silhouette_scores)]

best_k_file <- paste0("/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/best_k", run_suffix, ".txt")
writeLines(as.character(best_k), best_k_file)

#This function does not work in NaaVRE
#fviz_nbclust(pca_1_3_sample,FUNcluster = cluster::pam,method = 'silhouette', k.max = 8) 



In [ ]:
## GAP STATISTIC
#secondary check

#Read in layers df
pca_scores <- read.csv(pca_scores_file)
# Exclude the cell column from clustering
pca_1_3 <- as.matrix(
  pca_scores[, c("PC1", "PC2", "PC3")]
)


# Make a sample 
set.seed(123)
pca_1_3_sample <- pca_1_3[
  sample(seq_len(nrow(pca_1_3)), size = min(5000, nrow(pca_1_3))),
]


## Get gap statistic

set.seed(123)

gap_result <- cluster::clusGap(
  pca_1_3_sample,
  FUNcluster = cluster::pam,
  K.max = 8,
  B = 10,
  verbose = FALSE
)

plot(gap_result)

## Clustering (CLARA)

In [ ]:
# Read raster template
layers_veluwe <- terra::rast(layers_veluwe_file)

# Read selected number of clusters
best_k <- as.integer(readLines(best_k_file))

# Add a run suffix to keep outcomes of clustering with and without precipitation separate
run_suffix <- if (param_include_precipitation == 1L) "_with_precipitation" else "_soil_only"

# Cluster using only PC1–PC3, not the cell number
pca_scores <- read.csv(pca_scores_file)
pca_1_3 <- as.matrix(
  pca_scores[, c("PC1", "PC2", "PC3")]
)
## CLARA
set.seed(123)

# Running CLARA
clara <- cluster::clara(pca_1_3,  k = best_k, metric = "euclidean", stand = FALSE,
      samples = 50, pamLike = TRUE)

# Plot the cluster IDs back into the Veluwe
# Create empty output raster
ecoregions_rast <- layers_veluwe[[1]]
values(ecoregions_rast) <- NA 

ecoregions_rast[pca_scores$cell] <- clara$clustering#fills raster with cluster id
names(ecoregions_rast) <- "cluster_id"

plot(ecoregions_rast, main = paste("Candidate soil ecoregions, k =", best_k))

# Save output
ecoregions_datafile <- paste0("/home/jovyan/Cloud Storage/naa-vre-user-data/tmp/data/ecoregions_k", best_k,"clara", run_suffix, ".tif")
terra::writeRaster(ecoregions_rast,ecoregions_datafile, overwrite = TRUE)
